In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from tqdm import tqdm

# ─────────────────────────────────────────
# Config
# ─────────────────────────────────────────
BATCH_SIZE = 128
EPOCHS     = 10
LR         = 1e-3
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")

# ─────────────────────────────────────────
# Data — CIFAR-10
# ─────────────────────────────────────────
# ResNet expects 224x224, CIFAR-10 is 32x32 so we resize
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet mean (ResNet was pretrained on this)
        std =[0.229, 0.224, 0.225]    # ImageNet std
    )
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

# ─────────────────────────────────────────
# Model — ResNet50 with frozen backbone
# ─────────────────────────────────────────
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Freeze ALL layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer with linear classifier for 10 classes
# model.fc is the last layer in ResNet — feature_dim is 2048
feature_dim = model.fc.in_features          # 2048
model.fc    = nn.Linear(feature_dim, 10)    # only this will train

model = model.to(DEVICE)

# Verify — only model.fc parameters should be trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")

# ─────────────────────────────────────────
# Loss and Optimizer
# ─────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=LR)


Using device: cuda


100%|██████████| 170M/170M [00:04<00:00, 34.7MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 222MB/s]


Trainable params: 20,490 / 23,528,522  (0.09%)


In [2]:
def train_epoch(epoch):
    model.train()
    total_loss = 0
    correct    = 0
    total      = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")

    for batch_idx, (images, labels) in enumerate(loop):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total   += labels.size(0)

        loop.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{100 * correct / total:.2f}%"
        )

    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1} Train — Loss: {avg_loss:.4f} | Acc: {accuracy:.2f}%")
    return avg_loss, accuracy


In [3]:
def evaluate():
    model.eval()
    correct = 0
    total   = 0

    # Per-class accuracy tracking
    class_correct = [0] * 10
    class_total   = [0] * 10

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs        = model(images)
            _, predicted   = torch.max(outputs, 1)

            correct += (predicted == labels).sum().item()
            total   += labels.size(0)

            for i in range(labels.size(0)):
                label = labels[i].item()
                class_correct[label] += (predicted[i] == labels[i]).item()
                class_total[label]   += 1

    overall_acc = 100 * correct / total
    print(f"\nTest Accuracy: {overall_acc:.2f}%")
    print("\nPer-class accuracy:")
    for i, cls in enumerate(CLASSES):
        acc = 100 * class_correct[i] / class_total[i]
        print(f"  {cls:<12}: {acc:.2f}%")

    return overall_acc


In [4]:
print("\n" + "="*50)
print("Starting Linear Probe Training")
print("="*50 + "\n")

train_losses = []
train_accs   = []

for epoch in range(EPOCHS):
    loss, acc = train_epoch(epoch)
    train_losses.append(loss)
    train_accs.append(acc)

print("\n" + "="*50)
print("Final Evaluation on Test Set")
print("="*50)
evaluate()
# Save the linear layer weights only (tiny checkpoint)

torch.save(model.fc.state_dict(), 'linear_probe_weights.pth')
print("\nSaved linear layer weights to linear_probe_weights.pth")


Starting Linear Probe Training



Epoch 1 [Train]: 100%|██████████| 391/391 [02:51<00:00,  2.28it/s, acc=74.33%, loss=0.6317]


Epoch 1 Train — Loss: 0.7978 | Acc: 74.33%


Epoch 2 [Train]: 100%|██████████| 391/391 [02:57<00:00,  2.21it/s, acc=79.93%, loss=0.5229]


Epoch 2 Train — Loss: 0.5858 | Acc: 79.93%


...

Epoch 9 [Train]: 100%|██████████| 391/391 [02:57<00:00,  2.20it/s, acc=83.30%, loss=0.4681]


Epoch 9 Train — Loss: 0.4808 | Acc: 83.30%


Epoch 10 [Train]: 100%|██████████| 391/391 [02:57<00:00,  2.20it/s, acc=83.18%, loss=0.4161]

Epoch 10 Train — Loss: 0.4805 | Acc: 83.18%

Final Evaluation on Test Set



Test Accuracy: 83.03%

Per-class accuracy:
  airplane    : 84.00%
  automobile  : 90.40%
  bird        : 80.60%
  cat         : 77.30%
  deer        : 81.90%
  dog         : 73.90%
  frog        : 85.30%
  horse       : 81.20%
  ship        : 86.10%
  truck       : 89.60%

Saved linear layer weights to linear_probe_weights.pth
